# Bill&Meal: Receipt-to-Recipe Training

Trains a Gemma 4 E4B student model with QLoRA on receipt-recipe pairs.

**Before running:**
1. Runtime > Change runtime type > **T4 GPU** (or A100 if you have Pro)
2. In the left **Secrets** panel (🔑), add:
   - `HF_TOKEN` — HuggingFace access token, Read scope (https://huggingface.co/settings/tokens)
   - `WANDB_API_KEY` — wandb API key from https://wandb.ai/authorize (optional; if missing, wandb auto-disables)
3. Toggle **Notebook access** ON for both secrets.

## 1. Setup

In [ ]:
# Verify GPU
!nvidia-smi

In [ ]:
# Mount Google Drive (for checkpoints + outputs persistence across sessions)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repo and install package + ML dependencies.
REPO_URL = 'https://github.com/nik-pgh/bill_and_meal.git'
!git clone $REPO_URL /content/bill_and_meal
%cd /content/bill_and_meal
!pip install -q --upgrade pip
!pip install -e .
!pip install -q -r requirements.txt
!pip show bill_and_meal | head -3

# Make `bill_and_meal` importable in the running kernel without a restart:
#  1. PEP 660 .pth shim is only read at interpreter startup → add src/ to sys.path
#  2. Drop any stale `bill_and_meal*` entries cached from a prior failed import
#  3. Invalidate import finders so the new path is rescanned
import sys, importlib

SRC_PATH = '/content/bill_and_meal/src'
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

for _m in list(sys.modules):
    if _m == 'bill_and_meal' or _m.startswith('bill_and_meal.'):
        del sys.modules[_m]

importlib.invalidate_caches()

import bill_and_meal
print('bill_and_meal ready:', bill_and_meal.__file__)

In [ ]:
# Load secrets from Colab. HF_TOKEN env var alone authenticates
# huggingface_hub / transformers — no explicit login() call needed.
import os
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
try:
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
except Exception:
    os.environ['WANDB_DISABLED'] = 'true'
    print('WANDB_API_KEY not set — wandb reporting disabled.')

In [ ]:
# Sync data into Drive paths the colab.yaml config expects.
# The repo ships data/receipts/ and data/labeled/labeled_dataset.jsonl committed.
# Copy to Drive so checkpoints can resume across sessions.
import shutil
from pathlib import Path

drive_root = Path('/content/drive/MyDrive/bill_and_meal')
drive_root.mkdir(parents=True, exist_ok=True)

src_receipts = Path('/content/bill_and_meal/data/receipts')
dst_receipts = drive_root / 'receipts'
if not dst_receipts.exists():
    shutil.copytree(src_receipts, dst_receipts)

for filename in ('labeled/labeled_dataset.jsonl', 'manifest.jsonl'):
    src = Path('/content/bill_and_meal/data') / filename
    dst = drive_root / Path(filename).name
    if src.exists() and not dst.exists():
        shutil.copy2(src, dst)

print('Drive contents:', sorted(p.name for p in drive_root.iterdir()))

## 2. Verify Data and Config

In [ ]:
from bill_and_meal.config import load_config

config = load_config()
receipts_dir = Path(config['data']['receipts_dir'])
labeled_path = Path(config['data']['labeled_path'])

receipt_count = len(list(receipts_dir.glob('*'))) if receipts_dir.exists() else 0
labeled_count = sum(1 for _ in open(labeled_path)) if labeled_path.exists() else 0

print(f'Receipts:      {receipt_count}')
print(f'Labeled pairs: {labeled_count}')
print(f'Model:         {config["student"]["model"]}')
print(f'Epochs:        {config["training"]["epochs"]}')
print(f'Batch size:    {config["training"]["batch_size"]} (x {config["training"]["gradient_accumulation_steps"]} accum)')

assert labeled_count > 0, 'No labeled data found - check Drive paths.'

## 3. Train

In [ ]:
from bill_and_meal.train import run_training

run_training(config)

## 4. Quick Eval Sanity Check

In [ ]:
import json
from bill_and_meal.evaluate import ingredient_coverage

with open(config['data']['labeled_path']) as f:
    records = [json.loads(line) for line in f if line.strip()]

for r in records[:5]:
    cov = ingredient_coverage(r['ingredients'], r['teacher_output'])
    print(f"{r['id']}: coverage={cov:.1%}")

## 5. Export LoRA Adapter

In [ ]:
import shutil

output_dir = Path(config['training']['output_dir'])
if output_dir.exists():
    print(f'Model saved at: {output_dir}')
    print('Files:', [p.name for p in output_dir.iterdir()])
    archive = '/content/adapter'
    shutil.make_archive(archive, 'zip', output_dir)
    from google.colab import files
    files.download(f'{archive}.zip')
else:
    print('No model output found. Did training complete?')